# Interactive Modeling, Cross-Validation, and Ensembling
**Project**: House Price Analytics & Predictive Modeling

This notebook runs the complete, leak-free modeling pipeline. We set up path references to our modular Python files, train all candidate estimators using a strictly fold-fitted `FeatureEngineer`, perform Scipy SLSQP ensemble optimization, generate Kaggle submissions, and display SHAP explainability analyses.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Add src directory to path to enable clean modular imports
sys.path.append('../src')
from preprocessing import clean_data
from features import FeatureEngineer

%matplotlib inline
sns.set_theme(style='whitegrid')
print("Environment and imports loaded successfully.")

### 1. Load Data
Load the raw training dataset.

In [ ]:
train_df = pd.read_csv('../data/train.csv')
print(f"Loaded training data. Shape: {train_df.shape}")

### 2. Preprocessing & Skewness Correction
Remove outliers and apply the log1p transform to the target variable `SalePrice`.

In [ ]:
train_cleaned = clean_data(train_df, is_train=True)
y = np.log1p(train_cleaned['SalePrice'])
X_cleaned = train_cleaned.drop(columns=['SalePrice'])
print(f"Preprocessed dataset size: {X_cleaned.shape}, Target size: {y.shape}")

### 3. Leak-Free 5-Fold Feature Engineering & Model CV
We train all 7 models using 5-Fold Cross Validation. Crucially, the `FeatureEngineer` is fit and transformed inside the CV loop to guarantee zero target leakage.

In [ ]:
# Load tuned hyperparameters if available
tuned_params_path = '../outputs/models/best_tuned_params.pkl'
if os.path.exists(tuned_params_path):
    with open(tuned_params_path, 'rb') as f:
        tuned_params = pickle.load(f)
        # Extract parameters
        cat_params = tuned_params['CatBoost'][1]
        xgb_params = tuned_params['XGBoost'][1]
        lgb_params = tuned_params['LightGBM'][1]
else:
    cat_params = {'iterations': 1200, 'learning_rate': 0.03, 'depth': 5, 'random_seed': 42, 'verbose': 0}
    xgb_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 4, 'subsample': 0.8, 'colsample_bytree': 0.4, 'random_state': 42, 'verbosity': 0}
    lgb_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 4, 'num_leaves': 15, 'subsample': 0.8, 'colsample_bytree': 0.4, 'random_state': 42, 'verbose': -1}

models_dict = {
    'Ridge': Ridge(alpha=10.0),
    'Lasso': Lasso(alpha=0.0005, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.0005, l1_ratio=0.5, max_iter=10000),
    'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(**xgb_params),
    'LightGBM': LGBMRegressor(**lgb_params),
    'CatBoost': CatBoostRegressor(**cat_params)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_datasets = []

print("Preparing fold-specific feature sets (Strictly Leak-Free)...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_cleaned, y)):
    X_tr_raw = X_cleaned.iloc[train_idx].copy()
    X_va_raw = X_cleaned.iloc[val_idx].copy()
    y_tr_raw = train_cleaned['SalePrice'].iloc[train_idx]
    
    fold_engineer = FeatureEngineer()
    fold_engineer.fit(X_tr_raw, target=y_tr_raw)
    X_tr_fold = fold_engineer.transform(X_tr_raw)
    X_va_fold = fold_engineer.transform(X_va_raw)
    
    fold_datasets.append((X_tr_fold, X_va_fold, train_idx, val_idx))

oof_predictions = {}
cv_scores = {}

for model_name, model in models_dict.items():
    oof = np.zeros(len(X_cleaned))
    for fold, (X_tr_fold, X_va_fold, train_idx, val_idx) in enumerate(fold_datasets):
        y_tr = y.iloc[train_idx]
        from sklearn.base import clone
        fold_model = clone(model)
        fold_model.fit(X_tr_fold, y_tr)
        oof[val_idx] = fold_model.predict(X_va_fold)
        
    score = np.sqrt(mean_squared_error(y, oof))
    print(f"{model_name} OOF RMSLE: {score:.5f}")
    oof_predictions[model_name] = oof
    cv_scores[model_name] = score

### 4. Ensemble Weight Blending (SLSQP)
Optimize the blending weights using Scipy's Sequential Least Squares Programming (SLSQP).

In [ ]:
from scipy.optimize import minimize

candidate_models = ['CatBoost', 'LightGBM', 'ElasticNet', 'XGBoost']
available_models = [m for m in candidate_models if m in oof_predictions]
X_preds = np.column_stack([oof_predictions[m] for m in available_models])
target = y.values

def loss_func(weights):
    w = weights / np.sum(weights)
    pred = np.dot(X_preds, w)
    return np.sqrt(mean_squared_error(target, pred))

init_weights = np.ones(len(available_models)) / len(available_models)
bounds = [(0, 1)] * len(available_models)
constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
opt_weights = res.x / np.sum(res.x)

print("Optimized Ensemble Weights:")
for m, w in zip(available_models, opt_weights):
    print(f"  - {m}: {w:.4f}")
print(f"Optimized Ensemble OOF RMSLE: {np.sqrt(mean_squared_error(target, np.dot(X_preds, opt_weights))):.5f}")

### 5. Model Comparison Visual
Create a visual representation of how all models rank.

In [ ]:
scores_df = pd.DataFrame(list(cv_scores.items()), columns=['Model', 'CV_RMSLE'])
plt.figure(figsize=(10, 5))
sns.barplot(x='Model', y='CV_RMSLE', data=scores_df.sort_values('CV_RMSLE'), palette='viridis')
plt.title('Comparison of Model CV RMSLE (Lower is Better)', fontsize=16)
plt.ylabel('Out-Of-Fold RMSLE', fontsize=12)
plt.tight_layout()
plt.show()

### 6. Model Interpretation (SHAP)
Load a trained fold model and show global feature importances.

In [ ]:
import shap
print("Loading fold model for SHAP explainability...")
# Fit a global engineer to get consistent column mappings for display
global_eng = FeatureEngineer()
global_eng.fit(X_cleaned, target=train_cleaned['SalePrice'])
X_feat_global = global_eng.transform(X_cleaned)

model_to_explain = models_dict['LightGBM']
# Fit on global processed features for explanation convenience
model_to_explain.fit(X_feat_global, y)

explainer = shap.TreeExplainer(model_to_explain)
shap_values = explainer(X_feat_global)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_feat_global, show=False)
plt.title('SHAP Feature Importance Summary (Top 20 Features)', fontsize=14)
plt.tight_layout()
plt.show()